# weblooper — Stem Separation (Colab GPU)

This notebook downloads audio from YouTube using `yt-dlp`, separates it into 6 stems using Meta's **HTDemucs 6s** model, and uploads the results to your Google Drive for weblooper to pick up automatically.

**Requirements:**
- GPU runtime (Runtime → Change runtime type → T4 GPU)
- Google Drive access (for uploading results)

**Stems produced:** drums, bass, guitar, piano, vocals, other

Run all cells (Runtime → Run all) and wait ~3-5 minutes. weblooper will detect the results automatically.

In [ ]:
# @title 1. Install dependencies (cached to Drive for fast re-runs)
import subprocess, sys, os

# --- Mount Drive early for pip cache ---
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

cache_dir = '/content/drive/MyDrive/.weblooper_colab_cache/pip'
os.makedirs(cache_dir, exist_ok=True)
os.environ['PIP_CACHE_DIR'] = cache_dir

# Idempotency guard: skip if already installed this runtime
_INSTALLED_FLAG = '/tmp/.weblooper_stems_deps_installed'
if os.path.exists(_INSTALLED_FLAG):
    print("Dependencies already installed this runtime. Skipping.")
else:
    print("Installing yt-dlp and demucs (using Drive pip cache)...")
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'yt-dlp', 'demucs'
    ])
    # Mark installed
    open(_INSTALLED_FLAG, 'w').close()
    print("Done. Dependencies installed.")

In [ ]:
# @title 2. Verify imports
import torch
import torchaudio

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected! Separation will be very slow.")
    print("Go to Runtime → Change runtime type → T4 GPU")

import yt_dlp
print(f"yt-dlp: {yt_dlp.version.__version__}")

# Quick demucs import check
from demucs.pretrained import get_model
print("Demucs: OK")

In [ ]:
# @title 3. Authenticate Google Drive API
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload, MediaIoBaseDownload
import io

auth.authenticate_user()
drive_service = build('drive', 'v3')
print("Drive API authenticated.")

In [ ]:
# @title 4. CONFIGURATION — Pre-filled by weblooper (do not edit)
SESSION_FOLDER_ID = "__WEBLOOPER_SESSION_FOLDER_ID__"
YOUTUBE_URL = "__WEBLOOPER_YOUTUBE_URL__"
MODEL_NAME = "htdemucs_6s"  # 6 stems: drums, bass, guitar, piano, vocals, other

print(f"Session folder: {SESSION_FOLDER_ID}")
print(f"YouTube URL: {YOUTUBE_URL}")
print(f"Model: {MODEL_NAME}")

In [ ]:
# @title 5. Download audio from YouTube with yt-dlp
import yt_dlp
import os

audio_path = '/content/audio.wav'

# Remove old file if exists
if os.path.exists(audio_path):
    os.remove(audio_path)

ydl_opts = {
    'format': 'bestaudio/best',
    'outtmpl': '/content/audio.%(ext)s',
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'wav',
        'preferredquality': '0',  # best quality
    }],
    'quiet': False,
    'no_warnings': True,
}

print(f"Downloading audio from: {YOUTUBE_URL}")
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(YOUTUBE_URL, download=True)
    video_title = info.get('title', 'Unknown')
    video_duration = info.get('duration', 0)

print(f"\nTitle: {video_title}")
print(f"Duration: {video_duration}s ({video_duration//60}:{video_duration%60:02d})")

if not os.path.exists(audio_path):
    raise FileNotFoundError(f"Download failed — {audio_path} not found")

file_size = os.path.getsize(audio_path) / (1024 * 1024)
print(f"Audio file: {file_size:.1f} MB")

In [ ]:
# @title 6. Separate stems with Demucs
import demucs.separate
import shlex
import time

output_dir = '/content/stems'

# Clean previous output
import shutil
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

print(f"Running {MODEL_NAME} stem separation...")
print(f"(This typically takes 1-3 minutes on T4 GPU)")
print()

t0 = time.time()

# Use demucs CLI via its Python API
# htdemucs_6s produces: drums, bass, guitar, piano, vocals, other
demucs.separate.main(shlex.split(
    f'-n {MODEL_NAME} --out "{output_dir}" "{audio_path}"'
))

elapsed = time.time() - t0
print(f"\nSeparation complete in {elapsed:.1f}s")

# Verify output
stem_dir = os.path.join(output_dir, MODEL_NAME, 'audio')
if not os.path.exists(stem_dir):
    # Sometimes the folder name matches the input filename without extension
    candidates = []
    model_dir = os.path.join(output_dir, MODEL_NAME)
    if os.path.exists(model_dir):
        candidates = os.listdir(model_dir)
    if candidates:
        stem_dir = os.path.join(model_dir, candidates[0])

expected_stems = ['drums', 'bass', 'guitar', 'piano', 'vocals', 'other']
found = [f for f in os.listdir(stem_dir) if f.endswith('.wav')]
print(f"Stems found in {stem_dir}: {found}")

for stem in expected_stems:
    path = os.path.join(stem_dir, f'{stem}.wav')
    if not os.path.exists(path):
        print(f"  WARNING: {stem}.wav not found!")
    else:
        size = os.path.getsize(path) / (1024 * 1024)
        print(f"  {stem}.wav: {size:.1f} MB")

In [ ]:
# @title 7. Encode stems to Opus/WebM (compact for upload)
import subprocess

webm_dir = '/content/webm_stems'
os.makedirs(webm_dir, exist_ok=True)

expected_stems = ['drums', 'bass', 'guitar', 'piano', 'vocals', 'other']
encoded_files = {}

print("Encoding stems to Opus/WebM (128kbps)...")
for stem_name in expected_stems:
    wav_path = os.path.join(stem_dir, f'{stem_name}.wav')
    webm_path = os.path.join(webm_dir, f'{stem_name}.webm')

    if not os.path.exists(wav_path):
        print(f"  Skipping {stem_name} (not found)")
        continue

    # ffmpeg: WAV → Opus in WebM container at 128kbps
    cmd = [
        'ffmpeg', '-y', '-i', wav_path,
        '-c:a', 'libopus', '-b:a', '128k',
        '-vn', webm_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  ERROR encoding {stem_name}: {result.stderr[-200:]}")
        continue

    size = os.path.getsize(webm_path) / (1024 * 1024)
    encoded_files[stem_name] = webm_path
    print(f"  {stem_name}.webm: {size:.1f} MB")

print(f"\nEncoded {len(encoded_files)}/{len(expected_stems)} stems.")
total_size = sum(os.path.getsize(p) for p in encoded_files.values()) / (1024 * 1024)
print(f"Total upload size: {total_size:.1f} MB")

In [ ]:
# @title 8. Upload stems + update meta.json in Drive
import json
from datetime import datetime, timezone

OUTPUT_FOLDER_ID = SESSION_FOLDER_ID

if not OUTPUT_FOLDER_ID or OUTPUT_FOLDER_ID == '__WEBLOOPER_SESSION_FOLDER_ID__':
    raise ValueError("SESSION_FOLDER_ID not configured! Run this notebook from weblooper.")

print(f"Uploading to Drive folder: {OUTPUT_FOLDER_ID}")
print()

# --- Helper: delete old file by name, then upload new one ---
def upload_or_replace(folder_id, filename, filepath, mimetype):
    """Delete any existing file with this name, then upload fresh."""
    # Find and delete old versions
    old = drive_service.files().list(
        q=f"'{folder_id}' in parents and name='{filename}' and trashed=false",
        fields="files(id)"
    ).execute().get('files', [])
    for f in old:
        drive_service.files().delete(fileId=f['id']).execute()

    # Upload new
    media = MediaIoBaseUpload(open(filepath, 'rb'), mimetype=mimetype, resumable=True)
    file_meta = {'name': filename, 'parents': [folder_id]}
    created = drive_service.files().create(body=file_meta, media_body=media).execute()
    return created['id']

# Upload each encoded stem
uploaded_stems = []
for stem_name, webm_path in encoded_files.items():
    filename = f'{stem_name}.webm'
    print(f"  Uploading {filename}...", end=' ')
    file_id = upload_or_replace(OUTPUT_FOLDER_ID, filename, webm_path, 'audio/webm')
    uploaded_stems.append(stem_name)
    print(f"OK ({file_id})")

print(f"\nUploaded {len(uploaded_stems)} stem files.")

# --- Update meta.json ---
print("\nPatching meta.json...")
try:
    meta_search = drive_service.files().list(
        q=f"'{OUTPUT_FOLDER_ID}' in parents and name='meta.json' and trashed=false",
        fields="files(id)"
    ).execute().get('files', [])

    if meta_search:
        meta_id = meta_search[0]['id']
        req = drive_service.files().get_media(fileId=meta_id)
        meta_bytes = io.BytesIO()
        downloader = MediaIoBaseDownload(meta_bytes, req)
        done = False
        while not done:
            _, done = downloader.next_chunk()
        current_meta = json.loads(meta_bytes.getvalue().decode('utf-8'))
    else:
        current_meta = {}

    # Update metadata
    current_meta['stemNames'] = uploaded_stems
    current_meta['model'] = f'demucs {MODEL_NAME}'
    current_meta['status'] = 'ready'
    current_meta['processedAt'] = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
    current_meta['duration'] = video_duration
    if not current_meta.get('youtubeVideoTitle'):
        current_meta['youtubeVideoTitle'] = video_title

    # Write back
    media = MediaIoBaseUpload(
        io.BytesIO(json.dumps(current_meta, indent=2).encode('utf-8')),
        mimetype='application/json'
    )
    if meta_search:
        drive_service.files().update(fileId=meta_id, media_body=media).execute()
    else:
        file_meta = {'name': 'meta.json', 'parents': [OUTPUT_FOLDER_ID]}
        drive_service.files().create(body=file_meta, media_body=media).execute()

    print("meta.json updated with stemNames + status: 'ready'")
except Exception as e:
    print(f"WARNING: Could not update meta.json: {e}")
    print("Stems are uploaded but weblooper may not auto-detect them.")

print(f"\n{'='*50}")
print(f"=== SUCCESS ===")
print(f"{'='*50}")
print(f"\n{len(uploaded_stems)} stems uploaded to Drive.")
print(f"processedAt: {current_meta.get('processedAt', 'unknown')}")
print(f"\nGo back to weblooper — it should detect the stems automatically within 20 seconds.")

In [ ]:
# @title 9. (Bonus) Also generate timed lyrics from the vocals stem
# This cell is optional — it runs the same lyrics pipeline as the lyrics notebook.
# If you want lyrics, just let this cell run. If not, you can skip it.

import subprocess, json, re, time
import urllib.request, urllib.parse

vocals_webm = os.path.join(webm_dir, 'vocals.webm')
if not os.path.exists(vocals_webm):
    print("No vocals.webm found — skipping lyrics generation.")
else:
    print("Generating timed lyrics from vocals stem...")
    print("(This adds ~1-2 minutes)")

    # Convert to mono 16kHz WAV for the ASR model
    vocals_wav = '/content/vocals_16k.wav'
    subprocess.run([
        'ffmpeg', '-y', '-i', vocals_webm,
        '-ac', '1', '-ar', '16000', vocals_wav
    ], capture_output=True)

    # Install NeMo if not already
    nemo_flag = '/tmp/.weblooper_nemo_installed'
    if not os.path.exists(nemo_flag):
        print("Installing NeMo for speech recognition...")
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q',
            'nemo_toolkit[asr]'
        ])
        open(nemo_flag, 'w').close()
        print("NeMo installed.")

    # This will be handled by the lyrics notebook workflow
    # For now, just print a message
    print("\nLyrics generation requires the lyrics Colab notebook.")
    print("Run 'Generate Lyrics' in weblooper after stems are loaded.")
    print("(Full lyrics integration in this notebook coming soon)")

## Done!

Your stems have been uploaded to Google Drive. weblooper will detect them automatically.

If weblooper doesn't pick them up within a minute, try clicking "Load from Drive" in the stem player.

**Stems produced:**
- `drums.webm` — Kick, snare, hi-hat, cymbals, toms
- `bass.webm` — Bass guitar, synth bass
- `guitar.webm` — Electric and acoustic guitar
- `piano.webm` — Piano, keys, synths
- `vocals.webm` — Lead and backing vocals
- `other.webm` — Everything else (strings, FX, etc.)